# 📘 Assignment 04 — Advanced Agentic RAG with OpenAI APIs — **Solutions**

> ⚠️ This is the **reference solution** notebook. Work through `OpenAI_Advanced_RAG_Assignment.ipynb` first.

**Tasks covered:** 7 core tasks + 3 bonus challenges


---
## ⚙️ Installation

Notice the difference from the advanced notebook:  
- ✅ Add `langchain-openai` and `openai`  
- ❌ Remove `bitsandbytes`, `accelerate`, and `transformers` (no local model needed)
- ✅ Keep everything else

In [ ]:
# Run this cell once to install all required packages
!pip install langchain langchain-community langchain-openai
!pip install openai
!pip install chromadb
!pip install sentence-transformers   # still needed for the cross-encoder (V4)
!pip install rank-bm25
!pip install pypdf beautifulsoup4
!pip install python-dotenv           # recommended: load API key from .env file

print("✅ Installation complete")

---
## 1 — Import Required Libraries

In [ ]:
# Standard library
import os
import re
import warnings
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any

# ─── SOLUTION: Task 1 of 7 — Add the OpenAI imports ─────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv

# LangChain — document loading (unchanged from advanced notebook)
from langchain_community.document_loaders import PyPDFLoader, TextLoader, BSHTMLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# LangChain — vector store & retrieval (unchanged)
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# LangChain — agent framework (unchanged)
from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import Tool
from langchain.prompts import PromptTemplate

# Cost tracking (used in Bonus 1)
from langchain_community.callbacks import get_openai_callback

# Cross-encoder re-ranking stays local (no API needed)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

warnings.filterwarnings("ignore")
print("✅ Libraries imported")


---
## 2 — OpenAI API Setup

This section replaces the entire model-download block from the advanced notebook  
(quantization config, tokenizer, HuggingFace pipeline, LocalLLM wrapper — all gone).  
Instead you just need an API key and two constructor calls.

In [ ]:
# ─── SOLUTION: Task 2 of 7 — Load the API key ────────────────────────────────
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. "
        "Add it to a .env file in the project root: OPENAI_API_KEY=sk-..."
    )
print("✅ API key loaded")

# ─── SOLUTION: Task 3 of 7 — Initialise the chat model ───────────────────────
# Unlike the local pipeline, ChatOpenAI already implements LangChain's LLM
# interface — no wrapper class is needed.
llm = ChatOpenAI(
    model="gpt-4o-mini",   # cost-efficient; swap to "gpt-4o" for higher quality
    temperature=0,         # deterministic answers — good for RAG
    openai_api_key=api_key,
)

# ─── SOLUTION: Task 4 of 7 — Initialise embeddings ───────────────────────────
# text-embedding-3-small: 1536 dimensions, fast and cheap.
# text-embedding-3-large: 3072 dimensions, more accurate but ~6× more expensive.
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=api_key,
)

# Quick sanity checks
print(f"LLM type   : {type(llm).__name__}")
print(f"Embeddings : {type(embeddings).__name__}")
test_vec = embeddings.embed_query("hello world")
print(f"Embedding dims: {len(test_vec)}")


---
## 3 — Document Loading

> **✅ This section is identical to the Advanced RAG notebook.**  
> Copy the three document-loading functions from:  
> `Notebooks/03-AdvenceAgenticRAG/Advance_Agentic_RAG_with_Langchain.ipynb`  
> — Sections **5, 6, 7, and 8** (PDF, Text, HTML loaders + combine step).

Nothing changes here. The LangChain document loaders are model-agnostic.

In [ ]:
def extract_pdf_metadata(file_path: str) -> Dict[str, Any]:
    """
    Extract metadata from a PDF file.
    
    Args:
        file_path: Path to the PDF file
        
    Returns:
        Dictionary containing metadata
    """
    import PyPDF2
    from pathlib import Path
    
    metadata = {
        'source_type': 'pdf',
        'file_path': file_path,
        'file_name': Path(file_path).name,
        'file_size_kb': Path(file_path).stat().st_size / 1024,
        'modified_date': datetime.fromtimestamp(Path(file_path).stat().st_mtime).strftime('%Y-%m-%d'),
    }
    
    try:
        # Open PDF and extract metadata
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            pdf_metadata = pdf_reader.metadata
            
            # Extract standard metadata fields
            if pdf_metadata:
                metadata['pdf_title'] = pdf_metadata.get('/Title', Path(file_path).stem)
                metadata['pdf_author'] = pdf_metadata.get('/Author', 'Unknown')
                metadata['pdf_subject'] = pdf_metadata.get('/Subject', '')
                metadata['pdf_creator'] = pdf_metadata.get('/Creator', '')
                metadata['pdf_producer'] = pdf_metadata.get('/Producer', '')
                metadata['pdf_creation_date'] = pdf_metadata.get('/CreationDate', '')
            else:
                metadata['pdf_title'] = Path(file_path).stem
                metadata['pdf_author'] = 'Unknown'
            
            # Add page count
            metadata['total_pages'] = len(pdf_reader.pages)
            
            # Try to extract title from first page if not in metadata
            if not metadata.get('pdf_title') or metadata['pdf_title'] == Path(file_path).stem:
                first_page_text = pdf_reader.pages[0].extract_text()
                lines = first_page_text.split('\n')[:5]  # Check first 5 lines
                # Use first non-empty line as title
                for line in lines:
                    if line.strip() and len(line.strip()) > 10:
                        metadata['pdf_title'] = line.strip()
                        break
    
    except Exception as e:
        print(f"Warning: Could not extract full metadata from {file_path}: {e}")
        metadata['pdf_title'] = Path(file_path).stem
        metadata['pdf_author'] = 'Unknown'
    
    return metadata


def load_and_chunk_pdfs(
    file_paths: List[str],
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[Document]:
    """
    Load PDF files, extract metadata, and chunk them.
    
    Args:
        file_paths: List of paths to PDF files
        chunk_size: Size of each chunk
        chunk_overlap: Overlap between chunks
        
    Returns:
        List of Document objects with content and metadata
    """
    all_documents = []
    
    # Initialize text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    for file_path in file_paths:
        try:
            print(f"Processing PDF: {file_path}")
            
            # Extract metadata
            metadata = extract_pdf_metadata(file_path)
            
            # Load PDF content
            loader = PyPDFLoader(file_path)
            pages = loader.load()
            
            # Add page numbers to each page document
            for i, page in enumerate(pages):
                page.metadata.update(metadata)
                page.metadata['page_number'] = i + 1
            
            # Split into chunks
            chunks = text_splitter.split_documents(pages)
            
            # Add chunk index to metadata
            for chunk_idx, chunk in enumerate(chunks):
                chunk.metadata['chunk_index'] = chunk_idx
                chunk.metadata['total_chunks'] = len(chunks)
            
            all_documents.extend(chunks)
            print(f"  ✓ Created {len(chunks)} chunks from {len(pages)} pages")
        
        except Exception as e:
            print(f"  ✗ Error processing {file_path}: {e}")
    
    return all_documents

# Example usage (update paths to your PDF files)
pdf_files = [
    #"../UnstructureData/PDFFiles/Python Programming.pdf"
    # Add your PDF file paths here
    # "path/to/document1.pdf",
    # "path/to/document2.pdf",
]

# For demonstration, we'll check if there are PDFs in common locations
content_dirs = [
    Path.cwd() / "../UnstructureData/PDFFiles"
]

pdf_files = []
for content_dir in content_dirs:
    if content_dir.exists():
        pdf_files.extend([str(f) for f in content_dir.glob("*.pdf")])

if pdf_files:
    pdf_documents = load_and_chunk_pdfs(pdf_files[:3])  # Limit to first 3 PDFs
    print(f"\n✓ Total PDF chunks created: {len(pdf_documents)}")
else:
    pdf_documents = []
    print("\n⚠ No PDF files found. Add PDF file paths to pdf_files list.")

In [ ]:
def extract_text_metadata(file_path: str) -> Dict[str, Any]:
    """
    Extract metadata from a text file.
    
    Args:
        file_path: Path to the text file
        
    Returns:
        Dictionary containing metadata
    """
    from pathlib import Path
    
    metadata = {
        'source_type': 'text',
        'file_path': file_path,
        'file_name': Path(file_path).name,
        'file_size_kb': Path(file_path).stat().st_size / 1024,
        'modified_date': datetime.fromtimestamp(Path(file_path).stat().st_mtime).strftime('%Y-%m-%d'),
    }
    
    try:
        # Read the file to extract content-based metadata
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            content = file.read()
            lines = content.split('\n')
            
            # Try to extract title (first non-empty line or from filename)
            title = None
            for line in lines[:10]:  # Check first 10 lines
                if line.strip() and len(line.strip()) > 5:
                    # Check if it looks like a title (shorter, possibly with # for markdown)
                    clean_line = line.strip().lstrip('#').strip()
                    if len(clean_line) < 100:  # Titles are usually short
                        title = clean_line
                        break
            
            metadata['title'] = title if title else Path(file_path).stem
            
            # Try to find author information
            author = 'Unknown'
            author_patterns = [
                r'(?i)author[:\s]+([^\n]+)',
                r'(?i)by[:\s]+([^\n]+)',
                r'(?i)written by[:\s]+([^\n]+)',
            ]
            
            for pattern in author_patterns:
                match = re.search(pattern, content[:1000])  # Check first 1000 chars
                if match:
                    author = match.group(1).strip()
                    break
            
            metadata['author'] = author
            
            # Try to find date information
            date = None
            date_patterns = [
                r'(?i)date[:\s]+(\d{4}-\d{2}-\d{2})',
                r'(?i)published[:\s]+(\d{4}-\d{2}-\d{2})',
                r'(\d{4}-\d{2}-\d{2})',
            ]
            
            for pattern in date_patterns:
                match = re.search(pattern, content[:1000])
                if match:
                    date = match.group(1)
                    break
            
            metadata['publication_date'] = date if date else metadata['modified_date']
            
            # Add word count
            metadata['word_count'] = len(content.split())
            metadata['char_count'] = len(content)
    
    except Exception as e:
        print(f"Warning: Could not extract full metadata from {file_path}: {e}")
        metadata['title'] = Path(file_path).stem
        metadata['author'] = 'Unknown'
        metadata['publication_date'] = metadata['modified_date']
    
    return metadata


def load_and_chunk_text_files(
    file_paths: List[str],
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[Document]:
    """
    Load text files, extract metadata, and chunk them.
    
    Args:
        file_paths: List of paths to text files
        chunk_size: Size of each chunk
        chunk_overlap: Overlap between chunks
        
    Returns:
        List of Document objects with content and metadata
    """
    all_documents = []
    
    # Initialize text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    for file_path in file_paths:
        try:
            print(f"Processing text file: {file_path}")
            
            # Extract metadata
            metadata = extract_text_metadata(file_path)
            
            # Load text content
            loader = TextLoader(file_path, encoding='utf-8')
            documents = loader.load()
            
            # Add metadata to document
            for doc in documents:
                doc.metadata.update(metadata)
            
            # Split into chunks
            chunks = text_splitter.split_documents(documents)
            
            # Add chunk index to metadata
            for chunk_idx, chunk in enumerate(chunks):
                chunk.metadata['chunk_index'] = chunk_idx
                chunk.metadata['total_chunks'] = len(chunks)
            
            all_documents.extend(chunks)
            print(f"  ✓ Created {len(chunks)} chunks")
        
        except Exception as e:
            print(f"  ✗ Error processing {file_path}: {e}")
    
    return all_documents

# Find text files
text_files = []

content_dirs = [
    Path.cwd() / "../Samples/UnstructureData/TextFiles/python-3.14-docs-text"
]


for content_dir in content_dirs:
    if content_dir.exists():
        text_files.extend([str(f) for f in content_dir.glob("*.txt")])

# Also check the Lectures directory for transcripts
lecture_dirs = Path.cwd().parent.parent / "Lectures"
if lecture_dirs.exists():
    for week_dir in lecture_dirs.glob("Week-*"):
        text_files.extend([str(f) for f in week_dir.glob("*.txt")])

if text_files:
    text_documents = load_and_chunk_text_files(text_files[:5])  # Limit to first 5 text files
    print(f"\n✓ Total text chunks created: {len(text_documents)}")
else:
    text_documents = []
    print("\n⚠ No text files found. Add text file paths to text_files list.")

In [ ]:
def extract_html_metadata(file_path: str) -> Dict[str, Any]:
    """
    Extract metadata from an HTML file.
    
    Args:
        file_path: Path to the HTML file
        
    Returns:
        Dictionary containing metadata
    """
    from pathlib import Path
    from bs4 import BeautifulSoup
    
    metadata = {
        'source_type': 'html',
        'file_path': file_path,
        'file_name': Path(file_path).name,
        'file_size_kb': Path(file_path).stat().st_size / 1024,
        'modified_date': datetime.fromtimestamp(Path(file_path).stat().st_mtime).strftime('%Y-%m-%d'),
    }
    
    try:
        # Read and parse HTML
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            html_content = file.read()
            soup = BeautifulSoup(html_content, 'html.parser')
            
            # Extract title
            title = None
            # Try meta title first
            meta_title = soup.find('meta', {'name': 'title'}) or soup.find('meta', {'property': 'og:title'})
            if meta_title and meta_title.get('content'):
                title = meta_title.get('content')
            # Try title tag
            elif soup.title and soup.title.string:
                title = soup.title.string.strip()
            # Try h1 tag
            elif soup.h1:
                title = soup.h1.get_text().strip()
            
            metadata['title'] = title if title else Path(file_path).stem
            
            # Extract author
            author = 'Unknown'
            meta_author = soup.find('meta', {'name': 'author'}) or soup.find('meta', {'property': 'article:author'})
            if meta_author and meta_author.get('content'):
                author = meta_author.get('content')
            
            metadata['author'] = author
            
            # Extract description
            description = ''
            meta_desc = soup.find('meta', {'name': 'description'}) or soup.find('meta', {'property': 'og:description'})
            if meta_desc and meta_desc.get('content'):
                description = meta_desc.get('content')
            
            metadata['description'] = description
            
            # Extract keywords
            keywords = []
            meta_keywords = soup.find('meta', {'name': 'keywords'})
            if meta_keywords and meta_keywords.get('content'):
                keywords = [k.strip() for k in meta_keywords.get('content').split(',')]
            
            metadata['keywords'] = ', '.join(keywords) if keywords else ''
            
            # Extract publication date
            date = None
            date_meta = soup.find('meta', {'name': 'date'}) or \
                       soup.find('meta', {'property': 'article:published_time'}) or \
                       soup.find('meta', {'name': 'publish-date'})
            if date_meta and date_meta.get('content'):
                date = date_meta.get('content').split('T')[0]  # Extract just the date part
            
            metadata['publication_date'] = date if date else metadata['modified_date']
            
            # Extract URL if present
            url_meta = soup.find('meta', {'property': 'og:url'}) or soup.find('link', {'rel': 'canonical'})
            if url_meta:
                metadata['url'] = url_meta.get('content') or url_meta.get('href', '')
    
    except Exception as e:
        print(f"Warning: Could not extract full metadata from {file_path}: {e}")
        metadata['title'] = Path(file_path).stem
        metadata['author'] = 'Unknown'
        metadata['publication_date'] = metadata['modified_date']
    
    return metadata


def load_and_chunk_html_files(
    file_paths: List[str],
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[Document]:
    """
    Load HTML files, extract metadata, and chunk them.
    
    Args:
        file_paths: List of paths to HTML files
        chunk_size: Size of each chunk
        chunk_overlap: Overlap between chunks
        
    Returns:
        List of Document objects with content and metadata
    """
    all_documents = []
    
    # Initialize text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    for file_path in file_paths:
        try:
            print(f"Processing HTML file: {file_path}")
            
            # Extract metadata
            metadata = extract_html_metadata(file_path)
            
            # Load HTML content (UnstructuredHTMLLoader extracts text from HTML)
            loader = UnstructuredHTMLLoader(file_path)
            documents = loader.load()
            
            # Add metadata to document
            for doc in documents:
                doc.metadata.update(metadata)
            
            # Split into chunks
            chunks = text_splitter.split_documents(documents)
            
            # Add chunk index to metadata
            for chunk_idx, chunk in enumerate(chunks):
                chunk.metadata['chunk_index'] = chunk_idx
                chunk.metadata['total_chunks'] = len(chunks)
            
            all_documents.extend(chunks)
            print(f"  ✓ Created {len(chunks)} chunks")
        
        except Exception as e:
            print(f"  ✗ Error processing {file_path}: {e}")
    
    return all_documents

# Find HTML files
html_files = []

content_dirs = [
    Path.cwd() / "../Samples/UnstructureData/HTMLFiles/pandasDoc",
    Path.cwd() / "../Samples/UnstructureData/HTMLFiles/pandasDoc/user_guide",
]


for content_dir in content_dirs:
    if content_dir.exists():
        html_files.extend([str(f) for f in content_dir.glob("*.html")])
        html_files.extend([str(f) for f in content_dir.glob("*.htm")])

# Also check Projects directory
project_dir = Path.cwd().parent.parent / "Projects" / "Project 1 - DualLens Analytics"
if project_dir.exists():
    html_files.extend([str(f) for f in project_dir.glob("*.html")])

if html_files:
    html_documents = load_and_chunk_html_files(html_files[:3])  # Limit to first 3 HTML files
    print(f"\n✓ Total HTML chunks created: {len(html_documents)}")
else:
    html_documents = []
    print("\n⚠ No HTML files found. Add HTML file paths to html_files list.")

In [ ]:
# Combine all documents
all_documents = pdf_documents + text_documents + html_documents

print(f"Total document chunks from all sources: {len(all_documents)}")
print(f"  - PDF chunks: {len(pdf_documents)}")
print(f"  - Text chunks: {len(text_documents)}")
print(f"  - HTML chunks: {len(html_documents)}")

if len(all_documents) > 0:
    # Display sample metadata from first document
    print("\nSample metadata from first document:")
    sample_doc = all_documents[0]
    for key, value in sample_doc.metadata.items():
        print(f"  {key}: {value}")
    
    print(f"\nSample content (first 200 chars):")
    print(sample_doc.page_content[:200] + "...")
else:
    print("\n⚠ No documents loaded. Please add file paths in the previous cells.")
    print("Creating sample documents for demonstration...")
    
    # Create sample documents for demonstration
    sample_texts = [
        "Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. It enhances LLMs by providing relevant context from external knowledge bases.",
        "Agentic AI systems can reason, plan, and use tools autonomously. They go beyond simple prompt-response patterns to exhibit goal-directed behavior.",
        "Vector databases store embeddings and enable semantic search. They allow finding similar content based on meaning rather than just keyword matching.",
        "Cross-encoders evaluate query-document pairs jointly, providing more accurate relevance scores than bi-encoders which encode separately.",
        "Contextual compression extracts relevant information from retrieved documents, reducing noise and improving answer quality.",
    ]
    
    all_documents = [
        Document(
            page_content=text,
            metadata={
                'source_type': 'sample',
                'title': f'Sample Document {i+1}',
                'author': 'Demo',
                'chunk_index': i
            }
        )
        for i, text in enumerate(sample_texts)
    ]
    
    print(f"✓ Created {len(all_documents)} sample documents for demonstration")

---
## 4 — Create ChromaDB Vector Store

> **✅ This section is nearly identical to the Advanced RAG notebook.**  
> Copy Section 9 from the advanced notebook. The only difference is the
> `embeddings` variable — which now points to `OpenAIEmbeddings` instead
> of `HuggingFaceEmbeddings`. Because you named it `embeddings` in Task 4,
> no other change is required.

In [ ]:
# Create ChromaDB vector store
print("Creating ChromaDB vector store...")
print("This may take a few minutes depending on the number of documents...")

# Create vector store
vector_store = Chroma.from_documents(
    documents=all_documents,
    embedding=embedding_model,
    collection_name="advanced_agentic_rag",
    persist_directory="./chroma_db_advanced"
)

print(f"✓ Vector store created with {len(all_documents)} documents!")
print(f"✓ Persisted to: ./chroma_db_advanced")

# Test the vector store with a sample query
test_query = "What is Python function?"
test_results = vector_store.similarity_search(test_query, k=3)
print(f"\nTest query: '{test_query}'")
print(f"Retrieved {len(test_results)} documents")
print(f"\nTop result preview:")
print(test_results[0].page_content[:200] + "...")

---
## 5 — Create Base Retriever Tool

> **✅ Identical to the Advanced RAG notebook — copy Section 10.**

In [ ]:
# Create a retriever from the vector store
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}  # Retrieve top 5 most similar documents
)

# Test the retriever
test_docs = base_retriever.invoke("What is machine learning?")
print(f"Retriever test: Retrieved {len(test_docs)} documents")

# Create a retriever tool for the agent
def retriever_tool_func(query: str) -> str:
    """
    Retrieve relevant documents from the vector store based on a query.
    
    Args:
        query: The search query
        
    Returns:
        Formatted string with retrieved documents
    """
    try:
        docs = base_retriever.invoke(query)
        
        if not docs:
            return "No relevant documents found."
        
        # Format the results
        result = f"Retrieved {len(docs)} relevant documents:\n\n"
        for i, doc in enumerate(docs, 1):
            result += f"Document {i}:\n"
            result += f"Content: {doc.page_content[:300]}...\n"
            result += f"Source: {doc.metadata.get('title', doc.metadata.get('file_name', 'Unknown'))}\n"
            result += f"Type: {doc.metadata.get('source_type', 'Unknown')}\n"
            result += "-" * 80 + "\n"
        
        return result
    
    except Exception as e:
        return f"Error retrieving documents: {str(e)}"


# Create the tool
retriever_tool = Tool(
    name="document_retriever",
    func=retriever_tool_func,
    description=(
        "Useful for retrieving relevant information from the knowledge base. "
        "Input should be a search query or question. "
        "Returns relevant document chunks with their content and metadata."
    )
)

print("✓ Retriever tool created successfully!")
print(f"Tool name: {retriever_tool.name}")
print(f"Tool description: {retriever_tool.description}")

---
## 6 — Agent V1: Basic Retriever Agent

### What changes?
In the advanced notebook, building the agent looked like this:
```python
# Advanced notebook (local)
agent = create_react_agent(llm, tools, prompt)
```
In this assignment it is... exactly the same line. `ChatOpenAI` slots in  
because it implements the same LangChain interface that `LocalLLM` did.

The ReAct prompt template and `AgentExecutor` configuration are also unchanged.

In [ ]:
# ─── SOLUTION: Task 5 of 7 — Build Agent V1 ─────────────────────────────────
# Copied verbatim from Section 11 of the advanced notebook.
# The only requirement: llm is already ChatOpenAI (set in Task 3).

react_template = """Answer the following question as best you can. You have access to the following tools:

{tools}

Use this exact format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

IMPORTANT:
- Always write "Action:" on one line and "Action Input:" on the next line
- Do NOT combine them on the same line
- The Action must be exactly: document_retriever
- The Action Input should be a clear search query

Example:
Question: What is Python?
Thought: I need to search for information about Python
Action: document_retriever
Action Input: What is Python programming language
Observation: [results will appear here]
Thought: I now know the final answer
Final Answer: Python is a programming language...

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

react_prompt = PromptTemplate(
    template=react_template,
    input_variables=["input", "agent_scratchpad", "tools", "tool_names"]
)

agent_v1 = create_react_agent(
    llm=llm,
    tools=[retriever_tool],
    prompt=react_prompt,
)

agent_v1_executor = AgentExecutor(
    agent=agent_v1,
    tools=[retriever_tool],
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
    max_execution_time=60,
    return_intermediate_steps=True,
    early_stopping_method="generate",
)

print("✅ Agent V1 (Basic Retriever) created — using ChatOpenAI as the underlying LLM")


In [ ]:
# Define test prompts that will be reused across all agent versions
test_prompts = [
    "How do you define a function in Python and what are its key components?",
    #"What are the main differences between a list and a tuple in Python?",
    #"Can you explain the concept of inheritance in object-oriented programming?",
]

print("=" * 100)
print("AGENT V1 TESTING: Basic Retriever")
print("=" * 100)

# Store results for comparison
agent_v1_results = []

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*100}")
    print(f"Test Query {i}: {prompt}")
    print(f"{'='*100}\n")
    
    try:
        result = agent_v1_executor.invoke({"input": prompt})
        agent_v1_results.append(result)
        
        print(f"\n{'='*100}")
        print(f"FINAL ANSWER for Query {i}:")
        print(f"{'='*100}")
        print(result['output'])
        print(f"\n{'='*100}\n")
    
    except Exception as e:
        print(f"Error processing query {i}: {str(e)}")
        agent_v1_results.append({"error": str(e)})

print("\n" + "="*100)
print("AGENT V1 ANALYSIS")
print("="*100)
print("""
Agent V1 uses a simple retriever tool that performs vector similarity search.

Strengths:
✓ Fast and straightforward retrieval
✓ Good for queries that closely match document content
✓ Efficient with well-defined questions

Limitations:
✗ Single query approach - may miss relevant context
✗ No query enhancement or expansion
✗ Limited to semantic similarity only
✗ No re-ranking of results

Next: Agent V2 will add hypothetical question generation to broaden context retrieval.
""")

---
## 7 — Agent V2: Hypothetical Question Generation

### What changes?
In the advanced notebook, `generate_hypothetical_questions()` called the local
HuggingFace pipeline directly. With `ChatOpenAI`, the invocation style is different.

```python
# Advanced notebook — local pipeline call (do NOT copy this)
output = pipeline(prompt_text, max_new_tokens=150, ...)
response = output[0]["generated_text"]
```

```python
# OpenAI equivalent — use the LangChain interface (this is the pattern)
response = llm.invoke("your prompt here")
# response.content  ← this is the string you want
```

Everything else in V2 (the retriever tool wrapper, agent creation, executor) is the same.

In [ ]:
# ─── SOLUTION: Task 6 of 7 — Adapt generate_hypothetical_questions() ─────────
# Key difference from the advanced notebook:
#   llm.invoke(prompt) now returns an AIMessage object (not a plain string).
#   We must call .content to get the text before splitting on newlines.

import re

def generate_hypothetical_questions(query: str, llm, num_questions: int = 2) -> List[str]:
    """Generate related questions to broaden retrieval context."""
    prompt = f"""Given the following question, generate {num_questions} related questions that would help gather comprehensive information to answer it.

Original Question: {query}

Generate {num_questions} related questions (one per line):
1."""

    try:
        response = llm.invoke(prompt)
        text = response.content          # ← .content extracts the string from AIMessage
        lines = text.strip().split('\n')
        questions = []
        for line in lines:
            clean_line = re.sub(r'^\d+\.\s*', '', line.strip())
            if clean_line and len(clean_line) > 10:
                questions.append(clean_line)
        return questions[:num_questions]
    except Exception as e:
        print(f"Error generating hypothetical questions: {e}")
        return []

print("✅ generate_hypothetical_questions() defined (adapted for ChatOpenAI)")


In [ ]:
# Create enhanced retriever tool with hypothetical questions
def retriever_with_hypotheticals_func(query: str) -> str:
    """
    Retrieve documents using both the original query and hypothetical questions.
    
    Args:
        query: The search query
        
    Returns:
        Formatted string with retrieved documents from multiple queries
    """
    try:
        print(f"\n→ Original Query: {query}")
        
        # Generate hypothetical questions
        related_questions = generate_hypothetical_questions(query, llm, num_questions=2)
        
        if related_questions:
            print(f"→ Generated {len(related_questions)} hypothetical questions:")
            for i, q in enumerate(related_questions, 1):
                print(f"  {i}. {q}")
        
        # Retrieve documents for all questions
        all_docs = []
        seen_content = set()  # For deduplication
        
        # Retrieve for original query
        docs = base_retriever.invoke(query)
        for doc in docs:
            content_hash = hash(doc.page_content)
            if content_hash not in seen_content:
                seen_content.add(content_hash)
                all_docs.append(doc)
        
        # Retrieve for hypothetical questions
        for hyp_question in related_questions:
            docs = base_retriever.invoke(hyp_question)
            for doc in docs:
                content_hash = hash(doc.page_content)
                if content_hash not in seen_content:
                    seen_content.add(content_hash)
                    all_docs.append(doc)
        
        print(f"→ Retrieved {len(all_docs)} unique documents (after deduplication)")
        
        if not all_docs:
            return "No relevant documents found."
        
        # Format results (limit to top 7 to avoid overwhelming the agent)
        result = f"Retrieved {len(all_docs)} unique relevant documents:\n\n"
        for i, doc in enumerate(all_docs[:7], 1):
            result += f"Document {i}:\n"
            result += f"Content: {doc.page_content[:300]}...\n"
            result += f"Source: {doc.metadata.get('title', doc.metadata.get('file_name', 'Unknown'))}\n"
            result += f"Type: {doc.metadata.get('source_type', 'Unknown')}\n"
            result += "-" * 80 + "\n"
        
        return result
    
    except Exception as e:
        return f"Error retrieving documents: {str(e)}"


# Create the enhanced tool
retriever_with_hypotheticals_tool = Tool(
    name="enhanced_document_retriever",
    func=retriever_with_hypotheticals_func,
    description=(
        "Retrieves relevant information by generating related questions and searching for multiple perspectives. "
        "Input should be a search query or question. "
        "Returns diverse document chunks from multiple query angles."
    )
)

# Create Agent V2 with hypothetical questions
agent_v2 = create_react_agent(
    llm=llm,
    tools=[retriever_with_hypotheticals_tool],
    prompt=react_prompt
)

agent_v2_executor = AgentExecutor(
    agent=agent_v2,
    tools=[retriever_with_hypotheticals_tool],
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    return_intermediate_steps=True
)

print("✓ Agent V2 (with Hypothetical Questions) created successfully!")
print("\nEnhancements over V1:")
print("  + Generates related questions for broader context")
print("  + Retrieves documents from multiple query perspectives")
print("  + Deduplicates results for efficiency")
print("  + Provides more comprehensive information")

---
## 8 — Agent V3: Hybrid Search (BM25 + Vector)

> **✅ This section is identical to the Advanced RAG notebook.**  
> BM25 works on raw text — it has nothing to do with the LLM or embeddings.
> Copy Section 15 and 16 as-is.

In [ ]:
# Create BM25 retriever for keyword search
print("Creating BM25 retriever for keyword search...")

# BM25 needs the documents for indexing
bm25_retriever = BM25Retriever.from_documents(all_documents)
bm25_retriever.k = 5  # Return top 5 documents

print("✓ BM25 retriever created")

# Create ensemble retriever (hybrid search)
ensemble_retriever = EnsembleRetriever(
    retrievers=[base_retriever, bm25_retriever],
    weights=[0.5, 0.5]  # Equal weight to both retrievers
)

print("✓ Ensemble retriever created (50% vector + 50% BM25)")

# Test hybrid search
test_results = ensemble_retriever.invoke("machine learning algorithms")
print(f"Hybrid search test: Retrieved {len(test_results)} documents")

# Create hybrid retriever tool with hypothetical questions
def hybrid_retriever_with_hypotheticals_func(query: str) -> str:
    """
    Retrieve documents using hybrid search (vector + keyword) with hypothetical questions.
    
    Args:
        query: The search query
        
    Returns:
        Formatted string with retrieved documents
    """
    try:
        print(f"\n→ Original Query: {query}")
        
        # Generate hypothetical questions
        related_questions = generate_hypothetical_questions(query, llm, num_questions=2)
        
        if related_questions:
            print(f"→ Generated {len(related_questions)} hypothetical questions:")
            for i, q in enumerate(related_questions, 1):
                print(f"  {i}. {q}")
        
        # Retrieve documents using hybrid search
        all_docs = []
        seen_content = set()
        
        # Retrieve for original query using hybrid search
        print("→ Performing hybrid search (vector + BM25)...")
        docs = ensemble_retriever.invoke(query)
        for doc in docs:
            content_hash = hash(doc.page_content)
            if content_hash not in seen_content:
                seen_content.add(content_hash)
                all_docs.append(doc)
        
        # Retrieve for hypothetical questions
        for hyp_question in related_questions:
            docs = ensemble_retriever.invoke(hyp_question)
            for doc in docs:
                content_hash = hash(doc.page_content)
                if content_hash not in seen_content:
                    seen_content.add(content_hash)
                    all_docs.append(doc)
        
        print(f"→ Retrieved {len(all_docs)} unique documents via hybrid search")
        
        if not all_docs:
            return "No relevant documents found."
        
        # Format results
        result = f"Retrieved {len(all_docs)} documents using hybrid search (vector + keyword):\n\n"
        for i, doc in enumerate(all_docs[:7], 1):
            result += f"Document {i}:\n"
            result += f"Content: {doc.page_content[:300]}...\n"
            result += f"Source: {doc.metadata.get('title', doc.metadata.get('file_name', 'Unknown'))}\n"
            result += f"Type: {doc.metadata.get('source_type', 'Unknown')}\n"
            result += "-" * 80 + "\n"
        
        return result
    
    except Exception as e:
        return f"Error retrieving documents: {str(e)}"


# Create hybrid search tool
hybrid_search_tool = Tool(
    name="hybrid_document_retriever",
    func=hybrid_retriever_with_hypotheticals_func,
    description=(
        "Retrieves information using hybrid search (semantic vector search + keyword BM25 search) "
        "with hypothetical question generation. Combines semantic understanding with exact term matching. "
        "Input should be a search query or question."
    )
)

# Create Agent V3 with hybrid search
agent_v3 = create_react_agent(
    llm=llm,
    tools=[hybrid_search_tool],
    prompt=react_prompt
)

agent_v3_executor = AgentExecutor(
    agent=agent_v3,
    tools=[hybrid_search_tool],
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    return_intermediate_steps=True
)

print("\n✓ Agent V3 (with Hybrid Search) created successfully!")
print("\nEnhancements over V2:")
print("  + Hybrid search combining vector and keyword matching")
print("  + Better handling of technical terminology")
print("  + Balances semantic similarity with exact term matching")
print("  + More robust retrieval for diverse query types")

In [ ]:
print("=" * 100)
print("AGENT V3 TESTING: Hybrid Search Enhancement")
print("=" * 100)

agent_v3_results = []

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*100}")
    print(f"Test Query {i}: {prompt}")
    print(f"{'='*100}\n")
    
    try:
        result = agent_v3_executor.invoke({"input": prompt})
        agent_v3_results.append(result)
        
        print(f"\n{'='*100}")
        print(f"FINAL ANSWER for Query {i}:")
        print(f"{'='*100}")
        print(result['output'])
        print(f"\n{'='*100}\n")
    
    except Exception as e:
        print(f"Error processing query {i}: {str(e)}")
        agent_v3_results.append({"error": str(e)})

print("\n" + "="*100)
print("AGENT V3 ANALYSIS")
print("="*100)
print("""
Agent V3 adds hybrid search combining semantic and keyword matching.

Strengths:
✓ Combines semantic understanding (vectors) with exact matching (BM25)
✓ Better at finding specific terminology and concepts
✓ More robust across different query phrasings
✓ Effective for technical content with specific terms

Observations:
→ Retrieval balances meaning and keywords
→ Captures both paraphrased and exact mentions
→ More comprehensive coverage than single-method search

Limitations:
✗ All retrieved documents treated equally (no re-ranking)
✗ May still retrieve some less relevant content
✗ No prioritization of most pertinent results

Next: Agent V4 will add cross-encoder re-ranking to prioritize the most relevant results.
""")

---
## 9 — Cross-Encoder Re-Ranking Setup

> **✅ This section is identical to the Advanced RAG notebook.**  
> The `ms-marco-MiniLM-L-6-v2` cross-encoder runs locally — no API needed.  
> Copy Section 17 as-is.

In [ ]:
print("Loading cross-encoder model for re-ranking...")

# Model name for cross-encoder
cross_encoder_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Load cross-encoder model and tokenizer
cross_encoder_tokenizer = AutoTokenizer.from_pretrained(cross_encoder_model_name)
cross_encoder_model = AutoModelForSequenceClassification.from_pretrained(cross_encoder_model_name)

# Move to CPU (or GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
cross_encoder_model = cross_encoder_model.to(device)
cross_encoder_model.eval()  # Set to evaluation mode

print(f"✓ Cross-encoder model loaded on {device}")
print(f"Model: {cross_encoder_model_name}")

def rerank_documents(query: str, documents: List[Document], top_k: int = 5) -> List[Document]:
    """
    Re-rank documents using cross-encoder model.
    
    Args:
        query: Search query
        documents: List of documents to re-rank
        top_k: Number of top documents to return
        
    Returns:
        Re-ranked list of documents
    """
    if not documents:
        return []
    
    # Prepare query-document pairs
    pairs = [[query, doc.page_content] for doc in documents]
    
    # Tokenize and get scores
    with torch.no_grad():
        inputs = cross_encoder_tokenizer(
            pairs,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        scores = cross_encoder_model(**inputs).logits.squeeze(-1).cpu().numpy()
    
    # Sort documents by score
    scored_docs = list(zip(documents, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    
    # Return top-k documents
    reranked_docs = [doc for doc, score in scored_docs[:top_k]]
    
    # Print scores for debugging
    print(f"→ Re-ranking scores (top {min(top_k, len(scored_docs))}):")
    for i, (doc, score) in enumerate(scored_docs[:top_k], 1):
        print(f"  {i}. Score: {score:.4f} - {doc.metadata.get('title', 'Unknown')[:50]}")
    
    return reranked_docs

# Test re-ranking
print("\nTesting re-ranking...")
test_docs = ensemble_retriever.invoke("What is a Pandas dataframe?")
if test_docs:
    reranked = rerank_documents("What is a Pandas dataframe?", test_docs, top_k=3)
    print(f"✓ Re-ranked {len(test_docs)} documents, returned top {len(reranked)}")

---
## 10 — Agent V4: Hybrid Search + Re-Ranking

> **✅ This section is identical to the Advanced RAG notebook.**  
> Copy Sections 18 and 19 as-is.  
> The `rerank_documents()` function you copied above is called inside the
> retriever wrapper — nothing touches the LLM at this stage.

In [ ]:
# Create hybrid retriever with re-ranking tool
def hybrid_retriever_with_reranking_func(query: str) -> str:
    """
    Retrieve documents using hybrid search with hypothetical questions, then re-rank with cross-encoder.
    
    Args:
        query: The search query
        
    Returns:
        Formatted string with re-ranked documents
    """
    try:
        print(f"\n→ Original Query: {query}")
        
        # Generate hypothetical questions
        related_questions = generate_hypothetical_questions(query, llm, num_questions=2)
        
        if related_questions:
            print(f"→ Generated {len(related_questions)} hypothetical questions:")
            for i, q in enumerate(related_questions, 1):
                print(f"  {i}. {q}")
        
        # Retrieve documents using hybrid search
        all_docs = []
        seen_content = set()
        
        # Retrieve for original query
        print("→ Performing hybrid search...")
        docs = ensemble_retriever.invoke(query)
        for doc in docs:
            content_hash = hash(doc.page_content)
            if content_hash not in seen_content:
                seen_content.add(content_hash)
                all_docs.append(doc)
        
        # Retrieve for hypothetical questions
        for hyp_question in related_questions:
            docs = ensemble_retriever.invoke(hyp_question)
            for doc in docs:
                content_hash = hash(doc.page_content)
                if content_hash not in seen_content:
                    seen_content.add(content_hash)
                    all_docs.append(doc)
        
        print(f"→ Retrieved {len(all_docs)} unique documents")
        
        if not all_docs:
            return "No relevant documents found."
        
        # Re-rank documents using cross-encoder
        print("→ Re-ranking with cross-encoder...")
        reranked_docs = rerank_documents(query, all_docs, top_k=5)
        
        # Format results
        result = f"Retrieved and re-ranked {len(reranked_docs)} most relevant documents:\n\n"
        for i, doc in enumerate(reranked_docs, 1):
            result += f"Document {i} (Re-ranked):\n"
            result += f"Content: {doc.page_content[:300]}...\n"
            result += f"Source: {doc.metadata.get('title', doc.metadata.get('file_name', 'Unknown'))}\n"
            result += f"Type: {doc.metadata.get('source_type', 'Unknown')}\n"
            result += "-" * 80 + "\n"
        
        return result
    
    except Exception as e:
        return f"Error retrieving documents: {str(e)}"


# Create re-ranking tool
reranking_tool = Tool(
    name="reranking_document_retriever",
    func=hybrid_retriever_with_reranking_func,
    description=(
        "Retrieves and re-ranks documents using hybrid search (vector + keyword) with hypothetical questions, "
        "then uses a cross-encoder model to re-rank results by relevance. "
        "Returns the most relevant documents with highest quality. "
        "Input should be a search query or question."
    )
)

# Create Agent V4 with re-ranking
agent_v4 = create_react_agent(
    llm=llm,
    tools=[reranking_tool],
    prompt=react_prompt
)

agent_v4_executor = AgentExecutor(
    agent=agent_v4,
    tools=[reranking_tool],
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    return_intermediate_steps=True
)

print("\n✓ Agent V4 (with Re-Ranking) created successfully!")
print("\nEnhancements over V3:")
print("  + Cross-encoder re-ranking of retrieved documents")
print("  + More accurate relevance assessment")
print("  + Prioritizes most pertinent results")
print("  + Better quality top-k documents")

In [ ]:
print("=" * 100)
print("AGENT V4 TESTING: Cross-Encoder Re-Ranking Enhancement")
print("=" * 100)

agent_v4_results = []

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*100}")
    print(f"Test Query {i}: {prompt}")
    print(f"{'='*100}\n")
    
    try:
        result = agent_v4_executor.invoke({"input": prompt})
        agent_v4_results.append(result)
        
        print(f"\n{'='*100}")
        print(f"FINAL ANSWER for Query {i}:")
        print(f"{'='*100}")
        print(result['output'])
        print(f"\n{'='*100}\n")
    
    except Exception as e:
        print(f"Error processing query {i}: {str(e)}")
        agent_v4_results.append({"error": str(e)})

print("\n" + "="*100)
print("AGENT V4 ANALYSIS")
print("="*100)
print("""
Agent V4 adds cross-encoder re-ranking for superior result quality.

Strengths:
✓ Direct query-document relevance scoring
✓ More accurate than embedding similarity alone
✓ Prioritizes most relevant results effectively
✓ Better precision in top-k documents

Observations:
→ Re-ranking scores show clear relevance gradation
→ Top results are more focused and pertinent
→ Less noise in retrieved context

Limitations:
✗ Still includes full content of chunks (may have irrelevant parts)
✗ No extraction of specific relevant passages
✗ Agent still processes entire chunks

Next: Agent V5 will add contextual compression to extract only relevant content from chunks.
""")

---
## 11 — Contextual Compression Setup (V5)

### What changes?
`LLMChainExtractor.from_llm(llm)` is the same call as in the advanced notebook.  
The difference is what `llm` points to.

With a local 3B model, compression was unreliable — the model often returned  
garbled text or ignored the instruction to extract only the relevant part.  

With `gpt-4o-mini`, compression becomes a key strength: the model reliably  
strips irrelevant content and returns only the useful fragment.

In [ ]:
# ─── SOLUTION: Task 7 of 7 — Set up Contextual Compression ──────────────────
# Identical call to the advanced notebook.
# With ChatOpenAI, LLMChainExtractor automatically uses the API — no changes
# to the compression logic are needed.

print("Creating LLMChainExtractor for contextual compression...")

# Create the compressor
compressor = LLMChainExtractor.from_llm(llm)

print("✓ LLMChainExtractor created")

# Test the compressor
print("\nTesting contextual compression...")
test_docs = base_retriever.invoke("How to creating, reading, updating, and deleting ﬁles in Python?")
if test_docs:
    print(f"Original document length: {len(test_docs[0].page_content)} characters")
    
    # Compress the document
    compressed_docs = compressor.compress_documents(
        documents=test_docs[:1],
        query="How to creating, reading, updating, and deleting ﬁles in Python?"
    )
    
    if compressed_docs:
        print(f"Compressed document length: {len(compressed_docs[0].page_content)} characters")
        print(f"Compression ratio: {len(compressed_docs[0].page_content) / len(test_docs[0].page_content):.2%}")
        print(f"\nCompressed content preview:")
        print(compressed_docs[0].page_content[:300] + "...")

---
## 12 — Agent V5: Full Advanced Pipeline

> **✅ Mostly identical to the Advanced RAG notebook.**  
> Copy Sections 21 and 22 as-is.  
> The `compressor` variable (which you just created above) slots in
> directly — no changes to the agent structure.

In [ ]:
# Create complete advanced retriever with compression
def advanced_retriever_with_compression_func(query: str) -> str:
    """
    Complete advanced retrieval: hybrid search → hypothetical questions → re-ranking → compression.
    
    Args:
        query: The search query
        
    Returns:
        Formatted string with compressed, highly relevant content
    """
    try:
        print(f"\n→ Original Query: {query}")
        
        # Generate hypothetical questions
        related_questions = generate_hypothetical_questions(query, llm, num_questions=2)
        
        if related_questions:
            print(f"→ Generated {len(related_questions)} hypothetical questions:")
            for i, q in enumerate(related_questions, 1):
                print(f"  {i}. {q}")
        
        # Retrieve documents using hybrid search
        all_docs = []
        seen_content = set()
        
        # Retrieve for original query
        print("→ Performing hybrid search...")
        docs = ensemble_retriever.invoke(query)
        for doc in docs:
            content_hash = hash(doc.page_content)
            if content_hash not in seen_content:
                seen_content.add(content_hash)
                all_docs.append(doc)
        
        # Retrieve for hypothetical questions
        for hyp_question in related_questions:
            docs = ensemble_retriever.invoke(hyp_question)
            for doc in docs:
                content_hash = hash(doc.page_content)
                if content_hash not in seen_content:
                    seen_content.add(content_hash)
                    all_docs.append(doc)
        
        print(f"→ Retrieved {len(all_docs)} unique documents")
        
        if not all_docs:
            return "No relevant documents found."
        
        # Re-rank documents
        print("→ Re-ranking with cross-encoder...")
        reranked_docs = rerank_documents(query, all_docs, top_k=5)
        
        # Compress documents to extract only relevant content
        print("→ Compressing documents to extract relevant content...")
        compressed_docs = compressor.compress_documents(
            documents=reranked_docs,
            query=query
        )
        
        print(f"→ Compressed to {len(compressed_docs)} documents with relevant content")
        
        if not compressed_docs:
            # Fallback to original docs if compression fails
            compressed_docs = reranked_docs
        
        # Format results
        result = f"Retrieved, re-ranked, and compressed {len(compressed_docs)} most relevant passages:\n\n"
        for i, doc in enumerate(compressed_docs, 1):
            result += f"Passage {i} (Compressed & Re-ranked):\n"
            result += f"Content: {doc.page_content}\n"
            result += f"Source: {doc.metadata.get('title', doc.metadata.get('file_name', 'Unknown'))}\n"
            result += f"Type: {doc.metadata.get('source_type', 'Unknown')}\n"
            result += "-" * 80 + "\n"
        
        return result
    
    except Exception as e:
        return f"Error retrieving documents: {str(e)}"


# Create advanced tool with all enhancements
advanced_tool = Tool(
    name="advanced_document_retriever",
    func=advanced_retriever_with_compression_func,
    description=(
        "The most advanced retrieval tool combining: (1) hypothetical question generation, "
        "(2) hybrid search (vector + keyword), (3) cross-encoder re-ranking, and "
        "(4) LLM-based contextual compression. Returns only the most relevant extracted content. "
        "Input should be a search query or question."
    )
)

# Create Agent V5 - Complete Advanced RAG
agent_v5 = create_react_agent(
    llm=llm,
    tools=[advanced_tool],
    prompt=react_prompt
)

agent_v5_executor = AgentExecutor(
    agent=agent_v5,
    tools=[advanced_tool],
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    return_intermediate_steps=True
)

print("\n✓ Agent V5 (Complete Advanced RAG) created successfully!")
print("\n🎯 COMPLETE FEATURE SET:")
print("  ✓ Hypothetical question generation for broader context")
print("  ✓ Hybrid search (semantic + keyword) for robust retrieval")
print("  ✓ Cross-encoder re-ranking for relevance optimization")
print("  ✓ LLM-based contextual compression for focused content")
print("\nThis represents the state-of-the-art in RAG retrieval!")

In [ ]:
print("=" * 100)
print("AGENT V5 TESTING: Complete Advanced RAG with All Enhancements")
print("=" * 100)

agent_v5_results = []

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*100}")
    print(f"Test Query {i}: {prompt}")
    print(f"{'='*100}\n")
    
    try:
        result = agent_v5_executor.invoke({"input": prompt})
        agent_v5_results.append(result)
        
        print(f"\n{'='*100}")
        print(f"FINAL ANSWER for Query {i}:")
        print(f"{'='*100}")
        print(result['output'])
        print(f"\n{'='*100}\n")
    
    except Exception as e:
        print(f"Error processing query {i}: {str(e)}")
        agent_v5_results.append({"error": str(e)})

print("\n" + "="*100)
print("AGENT V5 FINAL ANALYSIS")
print("="*100)
print("""
Agent V5 represents the complete Advanced Agentic RAG system with all enhancements.

✨ COMPLETE FEATURE SET:
✓ Hypothetical question generation → Broader context
✓ Hybrid search (vector + BM25) → Robust retrieval
✓ Cross-encoder re-ranking → Relevance optimization  
✓ LLM-based contextual compression → Focused content

🎯 KEY IMPROVEMENTS:
✓ Highest quality retrieved context
✓ Most efficient use of context window
✓ Best balance of precision and recall
✓ Minimal noise and maximum signal
✓ Optimal answer quality and accuracy

📊 EVOLUTION SUMMARY:
V1: Basic retrieval → Good baseline
V2: + Hypothetical questions → Broader context
V3: + Hybrid search → Better term matching
V4: + Re-ranking → Higher precision
V5: + Compression → Cleanest context ⭐

This is the state-of-the-art approach for production RAG systems!
""")

---
## 13 — Reflection: Local vs Cloud

| Question | Observation |
|----------|-------------|
| Which was faster per query — local or API? | The OpenAI API is significantly faster (< 2 s vs 30–120 s for local Qwen2.5 on CPU). |
| Did answer quality improve with GPT-4o-mini? | Yes — answers are more coherent, better structured, and follow retrieval context more faithfully. |
| Was contextual compression (V5) more reliable with GPT-4o-mini? | Yes — the LLMChainExtractor produces cleaner extractions; local models often failed to follow the compression instruction. |
| What happens if the OpenAI API is unavailable? | The entire pipeline fails; building a local fallback (Agent V1 with Qwen) is a good resilience strategy. |
| For a privacy-sensitive use case, which would you choose? | Local model — data never leaves the machine; compliant with GDPR/HIPAA in on-premise deployments. |

---

## 🏆 Bonus Challenge Solutions


### Bonus 1 — Token Cost Tracking

`get_openai_callback()` is a LangChain context manager that accumulates token
usage across **all** LLM calls inside the `with` block.
`cb.total_cost` uses LangChain's built-in pricing table — always verify against
the current [OpenAI pricing page](https://openai.com/pricing) for production estimates.


In [ ]:
# Bonus 1 — Token cost tracking
def run_with_cost_tracking(executor, question: str):
    """Run an agent executor and print token usage + cost."""
    with get_openai_callback() as cb:
        result = executor.invoke({"input": question})
    print(f"Answer       : {result['output'][:200]}")
    print(f"Total tokens : {cb.total_tokens}")
    print(f"Prompt tokens: {cb.prompt_tokens}")
    print(f"Completion   : {cb.completion_tokens}")
    print(f"Total cost   : ${cb.total_cost:.6f}")
    return result, cb

result, cb = run_with_cost_tracking(agent_v1_executor, test_prompts[0])


### Bonus 2 — Model Comparison: gpt-4o-mini vs gpt-4o

`gpt-4o-mini` is ~20× cheaper per token than `gpt-4o`.
For most RAG tasks with good retrieval, quality is comparable.
Use `gpt-4o` when answers are consistently wrong or when multi-hop reasoning
over complex evidence is required.


In [ ]:
# Bonus 2 — Compare gpt-4o-mini vs gpt-4o on the same query
llm_mini = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=api_key)
llm_full = ChatOpenAI(model="gpt-4o",      temperature=0, openai_api_key=api_key)

comparison_question = test_prompts[0]
comparison_results = {}

for model_name, model_llm in [("gpt-4o-mini", llm_mini), ("gpt-4o", llm_full)]:
    agent = create_react_agent(llm=model_llm, tools=[retriever_tool], prompt=react_prompt)
    executor = AgentExecutor(
        agent=agent, tools=[retriever_tool], verbose=False,
        handle_parsing_errors=True, max_iterations=5,
    )
    with get_openai_callback() as cb:
        out = executor.invoke({"input": comparison_question})
    comparison_results[model_name] = {
        "answer": out["output"],
        "tokens": cb.total_tokens,
        "cost": cb.total_cost,
    }

for name, r in comparison_results.items():
    print(f"\n=== {name} ===")
    print(f"Answer : {r['answer'][:300]}")
    print(f"Tokens : {r['tokens']}  |  Cost: ${r['cost']:.6f}")


### Bonus 3 — LLM-Based Re-Ranking (replaces the cross-encoder)

**Cross-encoder**: processes all query-document pairs in a single batched forward
pass — fast and cheap at inference time, but requires a local GPU/CPU model.

**LLM re-ranking**: asks the model to score each document individually.
More flexible (understands nuance and intent) but slower and more expensive —
each document requires a separate API call.

**When to prefer LLM re-ranking:**
- You have ≤ 10 candidate documents (cost stays manageable)
- Queries require semantic reasoning beyond lexical similarity
- No GPU is available for the cross-encoder


In [ ]:
# Bonus 3 — Replace cross-encoder re-ranking with GPT-4o-mini scoring
def rerank_with_llm(query: str, documents: List, top_k: int = 5) -> List:
    """Score and re-rank documents using an LLM relevance judge."""
    scored = []
    for doc in documents:
        prompt = (
            "Rate how relevant the following document is to the query on a scale of 0–10.\n"
            "Reply with ONLY a single integer (e.g. 7).\n\n"
            f"Query: {query}\n\n"
            f"Document:\n{doc.page_content[:400]}\n\n"
            "Relevance score (0-10):"
        )
        try:
            response = llm.invoke(prompt)
            score = int(response.content.strip().split()[0])
        except (ValueError, IndexError):
            score = 0
        scored.append((doc, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    print(f"LLM re-ranking scores (top {top_k}): {[s for _, s in scored[:top_k]]}")
    return [doc for doc, _ in scored[:top_k]]

# Test — compare with cross-encoder results from Section 17
test_docs = ensemble_retriever.invoke("What is a Python decorator?")
reranked_llm = rerank_with_llm("What is a Python decorator?", test_docs, top_k=3)
print(f"\nTop result after LLM re-ranking:")
print(reranked_llm[0].page_content[:200])
